# 0. Importation 

In [4]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


# ============================================================
# PROJECT PATHS
# ============================================================

ROOT = Path('/home/phpham/Work/BioAgents')
OUTPUT_DIR = ROOT / "outputs" / "figures"
DEFAULT_COUNTS = ROOT / "outputs" / "reports" / "prisma_counts.json"


# ============================================================
# DRAWING UTILITIES
# ============================================================

COLORS = {
    "identification": "#DCEAF7",
    "screening": "#E6F2E6",
    "eligibility": "#FFF0D5",
    "included": "#EADFF2",
    "excluded": "#FCE3E3",
    "arrow": "#4A4A4A",
    "border": "#303030",
    "text": "#202020",
    "stage": "#4F5D75",
}


def add_box(
    ax,
    x: float,
    y: float,
    width: float,
    height: float,
    text: str,
    facecolor: str,
    fontsize: float = 10,
    linewidth: float = 1.2,
):
    """Add one rounded PRISMA box."""

    patch = FancyBboxPatch(
        (x - width / 2, y - height / 2),
        width,
        height,
        boxstyle="round,pad=0.012,rounding_size=0.015",
        linewidth=linewidth,
        edgecolor=COLORS["border"],
        facecolor=facecolor,
        zorder=2,
    )
    ax.add_patch(patch)

    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        color=COLORS["text"],
        linespacing=1.25,
        wrap=True,
        zorder=3,
    )


def add_arrow(
    ax,
    start: tuple[float, float],
    end: tuple[float, float],
    connectionstyle: str = "arc3",
):
    """Add a directional arrow between two boxes."""

    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=12,
        linewidth=1.25,
        color=COLORS["arrow"],
        connectionstyle=connectionstyle,
        shrinkA=0,
        shrinkB=0,
        zorder=1,
    )
    ax.add_patch(arrow)


def add_stage_label(ax, y: float, label: str):
    """Add the vertical stage label on the left."""

    ax.text(
        0.045,
        y,
        label,
        ha="center",
        va="center",
        rotation=90,
        fontsize=10,
        fontweight="bold",
        color="white",
        bbox={
            "boxstyle": "round,pad=0.45",
            "facecolor": COLORS["stage"],
            "edgecolor": COLORS["stage"],
        },
    )


def n(counts: dict, key: str) -> int:
    """Return an integer PRISMA count."""

    value = counts.get(key, 0)

    if value in (None, ""):
        return 0

    return int(value)


# ============================================================
# VALIDATION
# ============================================================

def validate_counts(counts: dict) -> None:
    """Check basic arithmetic consistency before drawing."""

    required = [
        "pubmed",
        "europe_pmc",
        "arxiv",
        "other_sources",
        "duplicates_removed",
        "automation_excluded",
        "records_screened",
        "records_excluded",
        "reports_sought",
        "reports_not_retrieved",
        "reports_assessed",
        "full_text_excluded",
        "system_entries",
        "evaluation_resources",
        "unique_publications",
    ]

    missing = [key for key in required if key not in counts]
    if missing:
        raise ValueError(
            "Missing keys in PRISMA count file: " + ", ".join(missing)
        )

    retrieved = (
        n(counts, "pubmed")
        + n(counts, "europe_pmc")
        + n(counts, "arxiv")
        + n(counts, "other_sources")
    )

    expected_screened = (
        retrieved
        - n(counts, "duplicates_removed")
        - n(counts, "automation_excluded")
    )

    if expected_screened != n(counts, "records_screened"):
        raise ValueError(
            "Inconsistent records_screened. "
            f"Expected {expected_screened}, "
            f"but received {n(counts, 'records_screened')}."
        )

    expected_sought = (
        n(counts, "records_screened")
        - n(counts, "records_excluded")
    )

    if expected_sought != n(counts, "reports_sought"):
        raise ValueError(
            "Inconsistent reports_sought. "
            f"Expected {expected_sought}, "
            f"but received {n(counts, 'reports_sought')}."
        )

    expected_assessed = (
        n(counts, "reports_sought")
        - n(counts, "reports_not_retrieved")
    )

    if expected_assessed != n(counts, "reports_assessed"):
        raise ValueError(
            "Inconsistent reports_assessed. "
            f"Expected {expected_assessed}, "
            f"but received {n(counts, 'reports_assessed')}."
        )

    if n(counts, "unique_publications") > n(counts, "reports_assessed"):
        raise ValueError(
            "unique_publications cannot exceed reports_assessed."
        )


# ============================================================
# PRISMA FIGURE
# ============================================================

def plot_prisma(counts: dict, output_stem: Path) -> None:
    """Create PRISMA-style flow diagram."""

    validate_counts(counts)

    retrieved = (
        n(counts, "pubmed")
        + n(counts, "europe_pmc")
        + n(counts, "arxiv")
        + n(counts, "other_sources")
    )

    fig, ax = plt.subplots(figsize=(12, 14))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    # Main-column geometry
    main_x = 0.43
    side_x = 0.79
    main_w = 0.50
    side_w = 0.31
    box_h = 0.075

    # Vertical coordinates
    y_identified = 0.915
    y_removed = 0.805
    y_screened = 0.675
    y_sought = 0.545
    y_assessed = 0.415
    y_included = 0.255
    y_final = 0.105

    # Stage labels
    add_stage_label(ax, 0.86, "IDENTIFICATION")
    add_stage_label(ax, 0.66, "SCREENING")
    add_stage_label(ax, 0.43, "ELIGIBILITY")
    add_stage_label(ax, 0.17, "INCLUDED")

    # Identification
    identified_text = (
        "Records identified from databases and sources\n"
        f"PubMed (n = {n(counts, 'pubmed'):,})\n"
        f"Europe PMC (n = {n(counts, 'europe_pmc'):,})\n"
        f"arXiv (n = {n(counts, 'arxiv'):,})\n"
        f"Other sources (n = {n(counts, 'other_sources'):,})\n"
        f"Total records (n = {retrieved:,})"
    )

    add_box(
        ax,
        main_x,
        y_identified,
        main_w,
        0.115,
        identified_text,
        COLORS["identification"],
        fontsize=9.5,
    )

    removed_text = (
        "Records removed before screening\n"
        f"Duplicate records removed (n = {n(counts, 'duplicates_removed'):,})\n"
        "Records excluded by automated retrieval rules "
        f"(n = {n(counts, 'automation_excluded'):,})"
    )

    add_box(
        ax,
        side_x,
        y_removed,
        side_w,
        0.095,
        removed_text,
        COLORS["excluded"],
        fontsize=9,
    )

    # Screening
    screened_text = (
        "Unique records screened by title and abstract\n"
        f"(n = {n(counts, 'records_screened'):,})"
    )

    add_box(
        ax,
        main_x,
        y_screened,
        main_w,
        box_h,
        screened_text,
        COLORS["screening"],
    )

    excluded_screening_text = (
        "Records excluded after title/abstract screening\n"
        f"(n = {n(counts, 'records_excluded'):,})"
    )

    add_box(
        ax,
        side_x,
        y_screened,
        side_w,
        box_h,
        excluded_screening_text,
        COLORS["excluded"],
        fontsize=9,
    )

    # Eligibility
    sought_text = (
        "Reports sought for retrieval or full-text assessment\n"
        f"(n = {n(counts, 'reports_sought'):,})"
    )

    add_box(
        ax,
        main_x,
        y_sought,
        main_w,
        box_h,
        sought_text,
        COLORS["eligibility"],
    )

    not_retrieved_text = (
        "Reports not retrieved\n"
        f"(n = {n(counts, 'reports_not_retrieved'):,})"
    )

    add_box(
        ax,
        side_x,
        y_sought,
        side_w,
        box_h,
        not_retrieved_text,
        COLORS["excluded"],
        fontsize=9,
    )

    assessed_text = (
        "Reports assessed for eligibility and FEV coding\n"
        f"(n = {n(counts, 'reports_assessed'):,})"
    )

    add_box(
        ax,
        main_x,
        y_assessed,
        main_w,
        box_h,
        assessed_text,
        COLORS["eligibility"],
    )

    exclusion_reasons = counts.get("full_text_exclusion_reasons", {})

    reason_lines = []
    for reason, value in exclusion_reasons.items():
        reason_lines.append(f"{reason} (n = {int(value):,})")

    exclusion_text = (
        "Reports excluded after full-text assessment\n"
        f"(n = {n(counts, 'full_text_excluded'):,})"
    )

    if reason_lines:
        exclusion_text += "\n" + "\n".join(reason_lines)

    exclusion_height = max(
        0.09,
        0.055 + 0.019 * len(reason_lines),
    )

    add_box(
        ax,
        side_x,
        y_assessed,
        side_w,
        exclusion_height,
        exclusion_text,
        COLORS["excluded"],
        fontsize=8.5,
    )

    # Included categories
    included_text = (
        "Entries included in the curated evidence map\n"
        f"Agentic or agent-adjacent system entries "
        f"(n = {n(counts, 'system_entries'):,})\n"
        f"Benchmark or evaluation resources "
        f"(n = {n(counts, 'evaluation_resources'):,})"
    )

    add_box(
        ax,
        main_x,
        y_included,
        main_w,
        0.095,
        included_text,
        COLORS["included"],
        fontsize=9.5,
    )

    overlap = (
        n(counts, "system_entries")
        + n(counts, "evaluation_resources")
        - n(counts, "unique_publications")
    )

    final_text = (
        "Unique publications represented in the final evidence map\n"
        f"(n = {n(counts, 'unique_publications'):,})\n"
        f"Publications contributing to both categories (n = {overlap:,})"
    )

    add_box(
        ax,
        main_x,
        y_final,
        main_w,
        0.085,
        final_text,
        COLORS["included"],
        fontsize=9.5,
        linewidth=1.7,
    )

    # Main vertical arrows
    add_arrow(
        ax,
        (main_x, y_identified - 0.058),
        (main_x, y_screened + box_h / 2),
    )
    add_arrow(
        ax,
        (main_x, y_screened - box_h / 2),
        (main_x, y_sought + box_h / 2),
    )
    add_arrow(
        ax,
        (main_x, y_sought - box_h / 2),
        (main_x, y_assessed + box_h / 2),
    )
    add_arrow(
        ax,
        (main_x, y_assessed - box_h / 2),
        (main_x, y_included + 0.047),
    )
    add_arrow(
        ax,
        (main_x, y_included - 0.047),
        (main_x, y_final + 0.043),
    )

    # Side arrows
    add_arrow(
        ax,
        (main_x + main_w / 2, 0.82),
        (side_x - side_w / 2, y_removed),
    )
    add_arrow(
        ax,
        (main_x + main_w / 2, y_screened),
        (side_x - side_w / 2, y_screened),
    )
    add_arrow(
        ax,
        (main_x + main_w / 2, y_sought),
        (side_x - side_w / 2, y_sought),
    )
    add_arrow(
        ax,
        (main_x + main_w / 2, y_assessed),
        (side_x - side_w / 2, y_assessed),
    )

    ax.text(
        0.5,
        0.985,
        "PRISMA-style flow diagram for literature identification,\n"
        "screening, eligibility assessment, and evidence-map inclusion",
        ha="center",
        va="top",
        fontsize=14,
        fontweight="bold",
        color=COLORS["text"],
    )

    fig.text(
        0.5,
        0.015,
        "Counts describe the curated structured narrative review and "
        "evidence map, not an exhaustive systematic review.",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color="#555555",
    )

    output_stem.parent.mkdir(parents=True, exist_ok=True)

    for extension in ("pdf", "png", "svg"):
        output_path = output_stem.with_suffix(f".{extension}")

        if extension == "png":
            fig.savefig(
                output_path,
                dpi=600,
                bbox_inches="tight",
                facecolor="white",
            )
        else:
            fig.savefig(
                output_path,
                bbox_inches="tight",
                facecolor="white",
            )

        print(f"Saved: {output_path}")

    plt.close(fig)


# ============================================================
# COMMAND-LINE ENTRY POINT
# ============================================================

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Generate a PRISMA-style review flow diagram."
    )
    parser.add_argument(
        "--counts",
        type=Path,
        default=DEFAULT_COUNTS,
        help="JSON file containing verified PRISMA counts.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=OUTPUT_DIR / "prisma_flow_diagram",
        help="Output path without a file extension.",
    )

    args = parser.parse_args()

    if not args.counts.exists():
        raise FileNotFoundError(
            f"PRISMA count file was not found: {args.counts}"
        )

    counts = json.loads(args.counts.read_text(encoding="utf-8"))
    plot_prisma(counts, args.output)


In [5]:
main()

usage: ipykernel_launcher.py [-h] [--counts COUNTS] [--output OUTPUT]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/1027/jupyter/runtime/kernel-v398192f1b3acb15eac92840d5038ef93bbdd2399b.json


SystemExit: 2

/home/phpham/miniconda3/envs/fev/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
